In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from openpmd_viewer import OpenPMDTimeSeries
from scipy.constants import c

In [ ]:
# Read in the cross-section data
cross_section_data = np.loadtxt(
    "../../../../warpx-data/MCC_cross_sections/H/H_on_H2_ionization.dat"
)
E_eV = 12e3
E_com_eV = E_eV * 2.0 / 3
# Find the cross-section at the center of mass energy
cross_section = np.interp(E_com_eV, cross_section_data[:, 0], cross_section_data[:, 1])
print(f"Cross-section at {E_com_eV} eV: {cross_section} m^2")

In [ ]:
ts = OpenPMDTimeSeries("./diags/diag")

In [ ]:
def plot_beam_current(iteration):
    """
    Plot the beam current for ions and neutral, as a function
    of the z position, averaged in time between iteration_start and iteration_end
    """
    # Loop over species
    for species, color in zip(["Hneutral", "Hplus"], ["b", "r"]):
        z, w, uz = ts.get_particle(
            ["z", "w", "uz"],
            species=species,
            iteration=iteration,
            select={"uz": [1e-3, None]},
        )
        w_binned, bins = np.histogram(z, bins=25, weights=w * uz)
        # Convert from number of particles per bin to beam current
        dz_bin = bins[1] - bins[0]
        beam_flux = w_binned / dz_bin * c
        plt.plot(bins[:-1], beam_flux, color=color, label=species)
    plt.legend(loc=0)
    plt.ylabel("Beam flux [A]")
    plt.xlabel("z [m]")

    # Plot theoretical curve
    n = 1e21
    flux = 1e20
    zmax = 0.2
    z = np.linspace(0, zmax, 100)
    plt.plot(z, flux * np.exp(-z * n * cross_section), color="b", ls=":")
    plt.plot(z, flux * (1 - np.exp(-z * n * cross_section)), color="r", ls=":")
    # plt.ylim(0, 45)

In [ ]:
plt.figure()
plot_beam_current(100)